# PatrolIQ - Notebook 1: Data Understanding and Cleaning

**Goal:** Load the Chicago crime dataset, understand its structure, and clean it up
before we do any analysis.

**Why we do this first:** Before building any model, we always need to check what
our data actually looks like - column names, data types, and missing values. Skipping
this step is one of the most common mistakes beginners make, since it can lead to
errors or wrong conclusions later on.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## Step 1: Load the Dataset

We downloaded a sample of 100,000 recent crime records from the official Chicago
Data Portal using their CSV export endpoint (`https://data.cityofchicago.org/resource/ijzp-q8t2.csv`)
with a `$limit` parameter. Downloading the full 7.8 million row dataset isn't practical
on a normal student laptop, so this sample is a realistic and honest compromise.

In [2]:
df = pd.read_csv("../data/chicago_crime_data.csv")
print("Shape:", df.shape)

Shape: (100000, 22)


## Step 2: First Look at the Data

In [3]:
df.head()

,id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_area,fbi_code,x_coordinate,y_coordinate,year,updated_on,latitude,longitude,location
0,14273182,JK346892,2026-07-25T00:00:00.000,091XX S CHAPPEL AVE,0760,BURGLARY,BURGLARY FROM MOTOR VEHICLE,STREET,False,False,413,4,7,48,06,1191438.0,1844734.0,2026,2026-08-01T15:41:35.000,41.728950,-87.574303,"\n, \n(41.728949593, -87.574303169)"
1,14274239,JK348199,2026-07-25T00:00:00.000,074XX W TALCOTT AVE,0486,BATTERY,DOMESTIC BATTERY SIMPLE,HOSPITAL BUILDING / GROUNDS,False,True,1612,16,41,10,08B,1125755.0,1938877.0,2026,2026-08-01T15:41:35.000,41.988631,-87.812829,"\n, \n(41.988630807, -87.812829371)"
2,14275478,JK349887,2026-07-25T00:00:00.000,106XX S INDIANA AVE,0810,THEFT,OVER $500,STREET,False,False,512,5,9,49,06,1179468.0,1834404.0,2026,2026-08-01T15:41:35.000,41.700884,-87.618466,"\n, \n(41.700884006, -87.618465883)"
3,14274031,JK347798,2026-07-25T00:00:00.000,052XX S FRANCISCO AVE,0486,BATTERY,DOMESTIC BATTERY SIMPLE,RESIDENCE,False,True,923,9,14,63,08B,1157930.0,1869772.0,2026,2026-08-01T15:41:35.000,41.798403,-87.696372,"\n, \n(41.798402937, -87.696372033)"
4,14273435,JK347277,2026-07-25T00:00:00.000,035XX W 80TH ST,1130,DECEPTIVE PRACTICE,FRAUD OR CONFIDENCE GAME,RESIDENCE,False,False,834,8,18,70,11,1154157.0,1851321.0,2026,2026-08-01T15:41:35.000,41.747846,-87.710698,"\n, \n(41.747846427, -87.710698363)"


In [4]:
df.columns.tolist()

['id',
 'case_number',
 'date',
 'block',
 'iucr',
 'primary_type',
 'description',
 'location_description',
 'arrest',
 'domestic',
 'beat',
 'district',
 'ward',
 'community_area',
 'fbi_code',
 'x_coordinate',
 'y_coordinate',
 'year',
 'updated_on',
 'latitude',
 'longitude',
 'location']

## Step 3: Check Data Types and Missing Values

`.info()` tells us the data type of each column and how many non-null values it has.
This is important because some columns (like `date`) are stored as plain text (`object`)
even though they represent dates, and we'll need to convert them later.

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    100000 non-null  int64  
 1   case_number           100000 non-null  object 
 2   date                  100000 non-null  object 
 3   block                 100000 non-null  object 
 4   iucr                  100000 non-null  object 
 5   primary_type          100000 non-null  object 
 6   description           100000 non-null  object 
 7   location_description  99614 non-null   object 
 8   arrest                100000 non-null  bool   
 9   domestic              100000 non-null  bool   
 10  beat                  100000 non-null  int64  
 11  district              100000 non-null  int64  
 12  ward                  100000 non-null  int64  
 13  community_area        100000 non-null  int64  
 14  fbi_code              100000 non-null  object 
 15  x

In [6]:
missing = df.isnull().sum()
missing[missing > 0]

location_description    386
x_coordinate             61
y_coordinate             61
latitude                 61
longitude                61
location                 61
dtype: int64

**What we found:** Only a small number of rows have missing values, mainly in
`location_description` (386 rows) and the coordinate columns (`latitude`, `longitude`,
`x_coordinate`, `y_coordinate`, `location` - all 61 rows). This is a very clean dataset
overall, likely because Chicago's official portal already does some quality checks
before publishing the data.

## Step 4: Data Cleaning

**Our cleaning plan:**
1. Drop rows with missing `latitude`/`longitude` - we cannot do geographic clustering
   without coordinates, and it's only 61 rows out of 100,000 (0.06%), so dropping them
   won't hurt our analysis.
2. Fill missing `location_description` with `"UNKNOWN"` - this is a categorical column,
   so filling with a placeholder category is simpler than dropping rows, and we don't
   lose the crime record itself.

**Why we're not doing more advanced imputation:** Techniques like KNN imputation or
regression-based imputation exist, but they add complexity that isn't necessary here
since the missing data is such a tiny fraction of the dataset. Simple dropping/filling
is the right level of effort for this amount of missing data.

In [7]:
df = df.dropna(subset=["latitude", "longitude"])
df["location_description"] = df["location_description"].fillna("UNKNOWN")

print("Shape after cleaning:", df.shape)
print("Remaining missing values:", df.isnull().sum().sum())

Shape after cleaning: (99939, 22)
Remaining missing values: 0


## Step 5: Feature Engineering - Temporal Features

The PDF requires us to extract temporal features from the `date` column: hour, day
of week, month, season, and a weekend flag. We also create a simple crime severity
score based on the crime type.

In [8]:
df["date"] = pd.to_datetime(df["date"])

df["hour"] = df["date"].dt.hour
df["day_of_week"] = df["date"].dt.day_name()
df["month"] = df["date"].dt.month

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Fall"

df["season"] = df["month"].apply(get_season)
df["is_weekend"] = df["day_of_week"].isin(["Saturday", "Sunday"])

df[["date", "hour", "day_of_week", "month", "season", "is_weekend"]].head()

,date,hour,day_of_week,month,season,is_weekend
0,2026-07-25,0,Saturday,7,Summer,True
1,2026-07-25,0,Saturday,7,Summer,True
2,2026-07-25,0,Saturday,7,Summer,True
3,2026-07-25,0,Saturday,7,Summer,True
4,2026-07-25,0,Saturday,7,Summer,True


## Step 6: Crime Severity Score

We built a simple manual severity scoring system (1 = low severity, 10 = high
severity) based on general knowledge of crime seriousness. This is **not** an
official or industry-standard scoring system - it's a reasonable approximation
we created ourselves for this project, and any crime type not explicitly listed
gets a default score of 3.

In [9]:
severity_map = {
    "HOMICIDE": 10,
    "CRIMINAL SEXUAL ASSAULT": 9,
    "CRIM SEXUAL ASSAULT": 9,
    "SEX OFFENSE": 8,
    "KIDNAPPING": 9,
    "ROBBERY": 7,
    "ASSAULT": 6,
    "BATTERY": 6,
    "WEAPONS VIOLATION": 7,
    "ARSON": 7,
    "BURGLARY": 5,
    "MOTOR VEHICLE THEFT": 5,
    "NARCOTICS": 4,
    "CRIMINAL DAMAGE": 4,
    "THEFT": 3,
    "DECEPTIVE PRACTICE": 3,
    "CRIMINAL TRESPASS": 3,
    "OFFENSE INVOLVING CHILDREN": 8,
    "INTIMIDATION": 5,
    "STALKING": 6,
    "PUBLIC PEACE VIOLATION": 2,
    "LIQUOR LAW VIOLATION": 2,
    "GAMBLING": 2,
    "OBSCENITY": 3,
    "INTERFERENCE WITH PUBLIC OFFICER": 4,
}

df["crime_severity_score"] = df["primary_type"].map(severity_map).fillna(3)
df[["primary_type", "crime_severity_score"]].drop_duplicates().sort_values("crime_severity_score", ascending=False)

,primary_type,crime_severity_score
582,HOMICIDE,10.0
1933,KIDNAPPING,9.0
71,CRIMINAL SEXUAL ASSAULT,9.0
168,SEX OFFENSE,8.0
212,OFFENSE INVOLVING CHILDREN,8.0
54,WEAPONS VIOLATION,7.0
79,ROBBERY,7.0
1174,ARSON,7.0
19,ASSAULT,6.0
627,STALKING,6.0


## Step 7: Save the Cleaned Data

We save this cleaned dataset so the next notebooks don't need to repeat this cleaning process.

In [10]:
df.to_csv("../outputs/cleaned_crime_data.csv", index=False)
print("Saved cleaned_crime_data.csv with shape:", df.shape)

Saved cleaned_crime_data.csv with shape: (99939, 28)
